# Reproducing Zhai et al. (2025) from dandiset 001538

**Paper.** Zhai S, Cui Q, Wokosin D, Sun L, Tkatch T, Crittenden JR, Graybiel AM, Surmeier DJ
(2025). State-dependent modulation of spiny projection neurons controls levodopa-induced
dyskinesia in a mouse model of Parkinson's disease. *Science Advances* 11, eadv8224.
[doi:10.1126/sciadv.adv8224](https://doi.org/10.1126/sciadv.adv8224). This attempt used the full
text (open access, PubMed Central PMC12674127).

**Dandiset.** [DANDI:001538](https://dandiarchive.org/dandiset/001538/0.260527.1302), version
0.260527.1302. The paper cites no DANDI version (its data statement points to a key resources table
on Zenodo). This is the only published version; the draft holds the same 468 files with the same
sizes and differs only in metadata (a link to the journal article among others), which the analysis
does not use.

**What the dandiset holds.** Patch-clamp recordings of somatic excitability and of the dendritic
back-propagation assay with two-photon line scans, for dSPNs and iSPNs in the LID off- and
on-states and under the drug and genetic manipulations of Figures 1, 3, 6, 7 and 9; GRAB-ACh3.0
imaging (one region per mouse, Figure 5); Sr2+-oEPSC recordings (one cell per mouse, Figures 2 and
4); AIM score tables for 9 mice and rotation videos (Figures 8 and 10); and raw spine image stacks
(Figures 2, 4, 6, 7 and 9). It holds no recordings from unlesioned or 6-OHDA-lesioned mice for the
patch-clamp figures, no spine counts, no rotation counts and no open-field data.

**Outside inputs.** The paper's per-cell and per-animal tables (Zenodo
doi:10.5281/zenodo.17307059) are used to test whether the dandiset's recordings are the paper's
cells (the link claims), for the genotype of the AIM mice, and for the claims on data the dandiset
does not hold (marked "elsewhere"). The analysis parameters come from the paper's analysis scripts
on Zenodo (doi:10.5281/zenodo.14145776, 14163502 and 14163731); see `lid.py`.

No random numbers are drawn anywhere in this notebook.

| Claim | Data | Status |
|---|---|---|
| link-somatic-cells | with inputs | tested |
| link-dendritic-cells | with inputs | tested |
| link-sr-cells | with inputs | tested |
| link-grab-rois | with inputs | tested |
| link-aim-mice | with labels | tested |
| fig1f-dspn-rheobase-on-lower | dandiset | tested |
| fig1e-dspn-fi-on-higher | dandiset | tested |
| fig1f-dspn-sch-no-change | dandiset | tested |
| fig1i-dspn-dendritic-on-higher | dandiset | tested |
| fig1i-dspn-dendritic-sch-lower | dandiset | tested |
| fig3d-ispn-rheobase-on-higher | dandiset | tested |
| fig3c-ispn-fi-on-lower | dandiset | tested |
| fig3d-ispn-sul-no-change | dandiset | tested |
| fig3f-ispn-dendritic-on-lower | dandiset | tested |
| fig3f-ispn-dendritic-sul-no-change | dandiset | tested |
| fig5f-ach-off-elevated | dandiset | tested |
| fig5f-ach-off-like-6ohda | dandiset | tested |
| fig6c-m1r-antag-somatic-unchanged | dandiset | tested |
| fig6d-m1r-antag-dendritic-lower | dandiset | tested |
| fig7c-cdgi-ko-somatic-oscillation | dandiset | tested |
| fig7f-cdgi-ko-dendritic-unchanged | dandiset | tested |
| fig9d-m1r-crispr-somatic-unchanged | dandiset | tested |
| fig2h, fig2i, fig4g, fig4h (Sr2+-oEPSC) | dandiset | blocked: one cell per mouse, 3 against 4 cells; the paper's test cannot reach P < 0.05 |
| fig5f-ach-6ohda-elevated, fig5f-ach-d2-control, fig5f-ach-da-suppresses | dandiset | blocked: 3 mice per group or 3 to 5 pairs; the paper's test cannot reach P < 0.05 |
| fig8bc-cdgi-aims, fig10b-m1r-aims | with labels | blocked: 1 wild-type mouse in Figure 8, 1 mouse per group in Figure 10 |
| fig2bc, fig4bc, fig4k, fig6f, fig7h, fig9f (spine density) | dandiset | blocked: raw stacks only; counting used AutoQuant, NeuronStudio or Imaris with manual correction |
| fig1-dspn-lesion-and-off-somatic, fig1-dspn-lesion-dendritic-unchanged, fig3-ispn-lesion-and-off-somatic, fig3-ispn-lesion-dendritic-up, fig7e-oxom-cdgi, fig8a-cdgi-rotations, fig10a-m1r-rotations, fig10ef-open-field | elsewhere | tested on the paper's tables |

The numbers for the blocked Sr2+-oEPSC, GRAB-ACh and AIM claims are shown under the link claims.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import lid

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = lid.asset_urls()
patch_items = [(p, u) for p, u in urls if "_icephys" in p and "+ogen" not in p]
sr_items = [(p, u) for p, u in urls if "_icephys+ogen" in p]
grab_items = [(p, u) for p, u in urls if p.endswith("_ophys.nwb")]
aim_items = [(p, u) for p, u in urls if "_ses-pan++ON" in p and not p.endswith("_image.nwb")]
video_items = [(p, u) for p, u in urls if p.endswith("_image.nwb")]
days = lid.load_many(lid.load_patch, patch_items, workers=12)
srs = lid.load_many(lid.load_sr, sr_items, workers=12)
grabs = lid.load_many(lid.load_grab, grab_items, workers=11)
aim = pd.concat(lid.load_many(lid.load_aim, aim_items, workers=12), ignore_index=True)
videos = pd.DataFrame(lid.load_many(lid.load_subject, video_items, workers=12))
tidy = lid.tidy_tables()
som = lid.somatic_table(days)
print(f"{len(urls)} NWB files listed at dandiset {lid.DANDISET_ID} version {lid.DANDISET_VERSION}; loaded "
      f"{len(days)} patch-clamp, {len(srs)} Sr2+-oEPSC, {len(grabs)} GRAB-ACh, {len(aim_items)} AIM and "
      f"{len(video_items)} video files in {time.time() - started:.0f} s")
print("Paper tables (Zenodo 17307059):", sorted(tidy))


def mw(a, b):
    """Two-sided Mann-Whitney test: n, medians, U, P and the rank-biserial effect (positive when b > a)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"n_a": len(a), "n_b": len(b), "median_a": np.median(a), "median_b": np.median(b),
            "U": u, "P": p, "effect_r": 1 - 2 * u / (len(a) * len(b))}


def show(d):
    """Rounded for printing; P to three significant digits."""
    out = {}
    for k, v in d.items():
        if isinstance(v, (float, np.floating)):
            out[k] = float(f"{v:.3g}") if k == "P" else round(float(v), 4)
        else:
            out[k] = int(v) if isinstance(v, np.integer) else v
    return out

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
def kind(p):
    if "_icephys+ogen" in p: return "Sr2+-oEPSC"
    if "_icephys" in p: return "somatic/dendritic patch"
    if p.endswith("_ophys.nwb"): return "GRAB-ACh imaging"
    if p.endswith("_image.nwb"): return "behaviour video"
    if "_ses-pan++ON" in p: return "AIM scores"
    if "confocal" in p: return "confocal spine stacks"
    return "2PLSM spine stacks"

inv = pd.DataFrame([{"kind": kind(p), **lid.parse_session(p)} for p, _ in urls])
print("NWB files by kind and session-name fields:")
print(inv.groupby(["kind", "cell_type", "state", "drug", "genotype"]).size().to_string())

The session names give cell type, state, drug and genotype. The dandiset holds patch-clamp files
for the LID off- and on-states, the drug and genetic manipulations, the GRAB-ACh imaging (3
unlesioned, 3 6-OHDA and 5 off-state mice), Sr2+-oEPSC cells, AIM tables, behaviour videos and
spine image stacks. There are no patch-clamp files for unlesioned control or 6-OHDA-lesioned mice
(the "control" groups of Figures 1 and 3), and no open-field data (Figure 10D to F).

In [ ]:
print("Somatic cells by group (condition_subfolder) against the paper's per-cell tables:")
rows = []
for g, spec in lid.TIDY_SOMATIC.items():
    s = som[som.group == g]
    rows.append({"group": g, "cells": len(s), "mice (dates)": s.date.nunique(),
                 "paper cells": len(lid.tidy_group(tidy, spec)),
                 "session drug labels": sorted(s.session_drug.unique())})
print(pd.DataFrame(rows).to_string(index=False))
print("\nCells with no group:", int((som.group == "?").sum()))
print("Cells whose session name says a drug but whose condition is baseline:")
print(som[(som.session_drug != "none") & ~som.condition.str.contains("with")][["date", "cell", "session_drug", "condition"]].to_string(index=False))
print("\nSweeps per cell and condition:", som.n_sweeps.value_counts().sort_index().to_dict())
print("Cells with a repeated step:", int((som.duplicates > 0).sum()),
      "; cells missing a step between 20 and 300 pA:", int(som[list(range(20, 301, 20))].isna().any(axis=1).sum()))
print(som[som[list(range(20, 301, 20))].isna().any(axis=1)][["group", "date", "cell"] + [a for a in range(20, 301, 20) if som[a].isna().any()]].to_string(index=False))
print(f"\nResting potential before the step (mV): {som.vrest_mV.min():.1f} to {som.vrest_mV.max():.1f}, median {som.vrest_mV.median():.1f}")
print(f"Largest difference between the stored stimulus and stimulus_current_pA (pA): {som.stim_error_pA.max():.3g}")
print(f"Action potentials inside the 0.2-0.7 s step: {som.spikes_in_step.sum()}; outside it: {som.spikes_outside_step.sum()} "
      f"(cells with any outside: {(som.spikes_outside_step > 0).sum()})")

Somatic sweeps are stored in volts at 10 kHz; resting potentials of -93 to -71 mV confirm the unit.
The stored stimulus of each somatic sweep equals the step amplitude in the index table, and the
step runs from 0.2 to 0.7 s (500 ms, as in the legends of Figures 3B and 6B). Nearly all action
potentials fall inside the step, so the voltage and the stimulus share a clock. The group of a cell
has to be taken from `condition_subfolder`: files named for SCH 23390 or sulpiride also hold
baseline (on-state) cells, and some cells were recorded in both conditions. Counted this way, every
group present matches the paper's n.

In [ ]:
den_sweeps = pd.DataFrame([{"date": d["date"], "cell_type": d["cell_type"], "cell": r["cell"], "condition": r["condition"],
                            "compartment": r["compartment"], "n_spikes": len(r["spike_times"]),
                            "first_spike_s": r["spike_times"][0] if len(r["spike_times"]) else np.nan,
                            "stim_max_abs_A": r["stim_max_abs_A"], "ls_ok": r["ls1_ok"] and r["ls2_ok"],
                            "rate": r["ch2_rate"], "n_samples": len(r["ch2"]), "md5": r["md5"]}
                           for d in days for r in d["dendritic"]])
print("Dendritic sweeps:", len(den_sweeps), "from", den_sweeps.groupby(["date", "cell"]).ngroups, "cells")
print("Spikes per sweep:", den_sweeps.n_spikes.value_counts().sort_index().to_dict())
print("First spike (s): median %.4f, range %.4f to %.4f" % (den_sweeps.first_spike_s.median(), den_sweeps.first_spike_s.min(), den_sweeps.first_spike_s.max()))
print("Largest absolute value of the stored dendritic stimulus (A):", den_sweeps.stim_max_abs_A.max())
print("Line-scan rate (Hz): %.1f to %.1f; samples per sweep: %s" % (den_sweeps.rate.min(), den_sweeps.rate.max(), den_sweeps.n_samples.value_counts().to_dict()))
print("Sweeps with no line-scan kymograph slice in the index table (reference -1 or past the data):", int((~den_sweeps.ls_ok).sum()), "of", len(den_sweeps))

sites = []
for d in days:
    if d["dendritic"]:
        s = lid.dendritic_sites(d)
        s["date"], s["cell_type"] = d["date"], d["cell_type"]
        s["group"] = [lid.GROUPS.get((d["cell_type"], c), "?") for c in s.condition]
        sites.append(s)
sites = pd.concat(sites, ignore_index=True)
print("\nSites:", len(sites), "; trials per site:", sites.n_trials.value_counts().sort_index().to_dict())
print("Area of dG/R0 from 0.95 to 1.5 s (after the spikes): median %.4f; the same window 1.5 s later: median %.4f; "
      "sites where the first is larger: %d of %d" % (sites.area.median(), sites.late_area.median(),
                                                      (sites.area > sites.late_area).sum(), len(sites)))
print("R0 (red, background subtracted) range %.0f to %.0f; G0 range %.0f to %.0f" % (sites.R0.min(), sites.R0.max(), sites.G0.min(), sites.G0.max()))

The dendritic assay evokes three spikes, the first at 1.00 s, and the Ca2+ signal rises after it:
the area in the analysis window is larger than in the same window 1.5 s later at nearly every site,
so the line scans and the voltage share the sweep clock. The stored dendritic stimulus is zero in
every sweep, so the injected current is not in the files; the three spikes show it was delivered.
The line-scan rate (678 to 693 Hz, 2500 lines per sweep) is the rate of the region series used
here. Some sweeps have no kymograph slice in the index table (count above; the kymographs hold
fewer trials than the region traces); the analysis uses the region traces (mean along the scan
line), which have a slice for every sweep.

In [ ]:
sr_rows = []
for s in srs:
    led = [w["led_trace_nonzero"] for w in s["sweeps"]]
    on = [w["led_onset_s"] for w in s["sweeps"] if w["led_trace_nonzero"]]
    pid = (s["date"][4:8] + s["date"][:4] if s["cell_type"] == "dSPN" else s["date"]) + s["session_letter"][0]
    sr_rows.append({"cell_type": s["cell_type"], "state": s["state"], "date": s["date"], "paper_id": pid,
                    "cell_number": s["cell_number"], "letter": s["session_letter"], "led_setting": s["led"],
                    "sweeps": len(s["sweeps"]), "n_vc": s["n_vc"], "n_og": s["n_og"],
                    "rate_kHz": sorted({round(w["rate"] / 1000) for w in s["sweeps"]}),
                    "sweeps_with_LED_trace": int(sum(led)),
                    "LED_onset_s": round(float(np.median(on)), 4) if on else np.nan, "epoch_stim_s": s["epoch_stim_s"],
                    "epoch_detect_s": s["epoch_detect"]})
sr_tab = pd.DataFrame(sr_rows)
print(sr_tab.to_string(index=False))

# Clock check: events in the detection window (40-400 ms after the pulse) against a window of the same
# length 560-920 ms after it, with the same detector.
ev_rows = []
for s in srs:
    for w in s["sweeps"]:
        onset = w["led_onset_s"] if w["led_trace_nonzero"] else s["epoch_stim_s"]
        e1 = lid.detect_events(w["current"], w["rate"], onset)
        e2 = lid.detect_events(w["current"], w["rate"], onset, window=(0.56, 0.92))
        ev_rows.append({"date": s["date"], "cell_type": s["cell_type"], "events_40_400": len(e1), "events_560_920": len(e2)})
ev = pd.DataFrame(ev_rows)
print("\nEvents per sweep, 40-400 ms after the pulse: %.2f; 560-920 ms after it: %.2f; cells with more in the first window: %d of %d"
      % (ev.events_40_400.mean(), ev.events_560_920.mean(),
         (ev.groupby("date").events_40_400.sum() > ev.groupby("date").events_560_920.sum()).sum(), ev.date.nunique()))

Each Sr2+-oEPSC file holds one cell (10 sweeps of 1 s), recorded at 10 or 20 kHz. The light
pulse is stored as an OptogeneticSeries per sweep, but in many sweeps that trace is all zero; where
it is not, the pulse starts at 19 to 20 ms, as the epochs table states (stimulation at 0.02 s,
detection from 0.06 to 0.42 s). Sweeps without a stored pulse use the epochs table's time. The
intracellular recordings table does not reference the stimulus (its stimulus rows are empty). Events
are more frequent after the pulse than late in the sweep, consistent with asynchronous release
evoked by the light, so the pulse time and the current share a clock. Each file's cell number and
session letter give an identifier of the form the paper uses (date and letter).

In [ ]:
g_rows = []
for m in grabs:
    gm = lid.grab_measures(m)
    sp = gm[gm.stimulation == "single_pulse"]
    g_rows.append({"state": m["state"], "date": m["date"], "regions": m["n_regions"], "frame_rate": m["frame_rate_2p"],
                   "treatments": sorted(sp.treatment), "reps": sp.n_reps.tolist(), "stim_rel_s": sorted(set(sp.stim_rel_s.round(2))),
                   "F0_range": (round(sp.F0.min()), round(sp.F0.max())), "Fmax": round(sp.Fmax.iloc[0]), "Fmin_TTX": round(sp.Fmin_ttx.iloc[0]),
                   "bg_reference_mean": round(m["bg_reference"]),
                   "auc_after_stim": round(float(sp[sp.treatment == "control"].auc_norm.iloc[0]), 3),
                   "auc_before_stim": round(float(sp[sp.treatment == "control"].auc_norm_pre.iloc[0]), 3)})
g_tab = pd.DataFrame(g_rows)
print(g_tab.to_string(index=False))

Each GRAB-ACh file holds one region of one slice per mouse, with two (sometimes three)
repetitions per treatment and stimulation, at 5.01 frames per second as the Methods state. The
stimulus time in the trial table is 10 s after the trial start in five files and 3 s in six; the
analysis uses the table's time, where the paper's script fixes 10 s. Without drug, the normalised
area after the stimulus is several times the area of the 2 s before it in every mouse, so the trial
table's stimulus time and the traces share a clock. The background reference images are described
as taken at zero laser power, but their mean (381 to 525) is above the signal itself (F0 179 to
355), so they cannot be the PMT background; the normalised area does not depend on a constant
background, so this does not affect Figure 5F.

In [ ]:
aim["animal_id"] = aim.animal_id.astype(int)
print("AIM rows by animal, the table's genotype column, the subject genotype and the session name's genotype:")
aim["session_genotype"] = [lid.parse_session(p)["genotype"] for p in aim.path]
print(aim.groupby(["animal_id", "genotype", "subject_genotype", "session_genotype"]).agg(
    rows=("total_score", "size"), sessions=("session_number", lambda x: sorted(set(x)))).to_string())
print("\nTime points (min after levodopa):", aim.time_minutes.value_counts().sort_index().to_dict())
for c in ["axial_score", "limb_score", "orolingual_score", "total_score"]:
    print(f"{c}: {aim[c].min()} to {aim[c].max()}")
bad = aim[~np.isclose(aim.total_score, aim.axial_score + aim.limb_score + aim.orolingual_score)]
print("Rows where total differs from axial + limb + orolingual:", len(bad), "of", len(aim))
print(bad[["animal_id", "session_number", "time_minutes", "axial_score", "limb_score", "orolingual_score", "total_score"]].to_string(index=False))
videos["animal"] = videos.subject_id.str.extract(r"(\d+)$")[0]
print("\nBehaviour video files (rotation videos; no counts): subject genotype by animal")
print(videos[["animal", "genotype", "subject_id"]].to_string(index=False))

AIM scores are stored per time point (0 to 4 for each of axial, limb and orolingual, every 20
min), and the total is their sum. The genotype labels contradict each other: in the CDGI study every
subject is "Wild-type CDGI" and every session name says WT, while the table's genotype column says
"CDGI Knockout" for some sessions, "Wild Type" or "unknown" for others, changing within an animal,
and the behaviour video files of the same animals (1945, 2519, 2587, 2828, 6572) say "CDGI KO". The
genotype of these mice cannot be read from the dandiset. The videos carry no rotation counts.

In [ ]:
dups = som[som.duplicated("md5", keep=False)]
print("Somatic cells whose voltage traces are identical to another cell's:", len(dups))
print(dups[["group", "date", "cell", "condition", "md5"]].to_string(index=False) if len(dups) else "")
same_fi = som[som.duplicated(list(range(20, 301, 20)), keep=False)].sort_values(list(range(20, 301, 20)))
print("\nCells with identical spike counts at every step (not identical traces):")
print(same_fi[["group", "date", "cell", "md5"]].to_string(index=False))
dd = den_sweeps[den_sweeps.duplicated("md5", keep=False)]
print("\nDendritic voltage sweeps identical to another sweep:", len(dd))
sr_md5 = pd.Series([w["md5"] for s in srs for w in s["sweeps"]])
print("Sr2+-oEPSC sweeps identical to another sweep:", int(sr_md5.duplicated(keep=False).sum()))
aim_keys = aim.groupby(["animal_id", "session_number"]).total_score.apply(tuple)
print("AIM sessions with the same score sequence as another animal's session:",
      int(aim_keys.reset_index().assign(a=lambda x: x.total_score.astype(str)).duplicated("a", keep=False).sum()))

No recording is deposited twice: no two somatic cells, dendritic sweeps or Sr2+-oEPSC sweeps have
identical traces. An iSPN on-state cell (20160609, cell 1) and a CDGI KO on-state cell (20160913,
cell 1) have identical spike counts at every step but different traces, so they are different
cells. No two AIM sessions have the same score sequence.

## Links to the paper's per-cell tables

The paper deposited per-cell and per-animal tables (Zenodo doi:10.5281/zenodo.17307059). These
sections test whether the dandiset's recordings are the paper's cells; the claims on the dandiset
below use only the dandiset.

### link-somatic-cells

> The dandiset's somatic current-step recordings are those behind the paper's somatic excitability
> figures, and the spike counts computed from them agree with the paper's per-cell tables.

In [ ]:
amps = list(range(20, 301, 20))
link_rows = []
for g, spec in lid.TIDY_SOMATIC.items():
    paper = lid.tidy_fi(lid.tidy_group(tidy, spec))
    P = paper[amps].to_numpy(float)
    prheo = [next((a for a, v in zip(amps, r) if v >= 1), np.nan) for r in P]
    for _, c in som[som.group == g].iterrows():
        v = c[amps].to_numpy(float)
        ok = ~np.isnan(v)
        dist = np.abs(P[:, ok] - v[ok]).sum(axis=1)
        j = int(np.argmin(dist))
        link_rows.append({"group": g, "date": c.date, "cell": c.cell, "best_row": paper["cell id"].iloc[j],
                          "spike_diff": dist[j], "exact": dist[j] == 0, "n_exact_rows": int((dist == 0).sum()),
                          "rheobase": c.rheobase, "paper_rheobase": prheo[j]})
link_som = pd.DataFrame(link_rows)
summary = link_som.groupby("group").agg(cells=("cell", "size"), exact=("exact", "sum"),
                                        within3=("spike_diff", lambda x: int((x <= 3).sum())),
                                        rheobase_equal=("rheobase", "size"))
summary["rheobase_equal"] = link_som.assign(eq=link_som.rheobase == link_som.paper_rheobase).groupby("group").eq.sum()
summary["paper_n"] = [len(lid.tidy_group(tidy, lid.TIDY_SOMATIC[g])) for g in summary.index]
print(summary.to_string())
n = len(link_som)
print(f"\nAll groups: {n} cells; exact match {link_som.exact.sum()} ({link_som.exact.mean():.1%}); "
      f"best match within 3 spikes {(link_som.spike_diff <= 3).sum()} ({(link_som.spike_diff <= 3).mean():.1%}); "
      f"rheobase equal to the best match {(link_som.rheobase == link_som.paper_rheobase).sum()}")
print("Groups whose cell count equals the paper's n:", int((summary.cells == summary.paper_n).sum()), "of", len(summary))
print("\nCells without an exact match:")
print(link_som[~link_som.exact].to_string(index=False))

**Result: partially reproduced.** Every group the dandiset holds has the paper's n (12 of 12 groups, 123 cells). 90 of 123 cells (73.2%) match a row of the paper's table exactly over 20 to 300 pA and 105 (85.4%) within 3 spikes, below the 80% and 95% the criterion asks for; 108 have the same rheobase as their best match. The mismatches fall in most groups, most often in dSPN on (6 of 15 exact).

### link-dendritic-cells

> The dandiset's dendritic line scans are those behind the paper's dendritic excitability index
> figures, and the index computed from them agrees with the paper's per-cell values.

In [ ]:
def cell_index(sites, basal_range=lid.BASAL_RATIO_RANGE):
    out = []
    for (date, ct), s in sites.groupby(["date", "cell_type"]):
        di = lid.dendritic_index(s, basal_range)
        di["date"], di["cell_type"] = date, ct
        out.append(di)
    out = pd.concat(out, ignore_index=True)
    out["group"] = [lid.GROUPS.get((ct, c), "?") for ct, c in zip(out.cell_type, out.condition)]
    return out.sort_values(["group", "date", "cell"]).reset_index(drop=True)

dend = cell_index(sites)
print("Cells with dendritic sites:", len(dend), "; with an index:", int(dend["index"].notna().sum()))
print("Cells without an index (no proximal and distal site of the same number):")
print(dend[dend["index"].isna()][["group", "date", "cell", "pairs", "unpaired_sites"]].to_string(index=False))
print("Pairs:", int(dend.pairs.sum()), "; dropped by the basal-ratio rule:", int((dend.pairs - dend.pairs_kept).sum()))

rows, pairs_rows = [], []
for g, spec in lid.TIDY_DENDRITIC.items():
    paper = lid.tidy_group(tidy, spec)
    pv = paper.iloc[:, -1].to_numpy(float)
    mine = dend[(dend.group == g) & dend["index"].notna()]
    rows.append({"group": g, "cells": len(mine), "paper_n": len(paper), "median": mine["index"].median(), "paper_median": np.median(pv)})
    if g.startswith("Fig6"):
        ids = paper.iloc[:, 0].astype(str)
        for _, c in mine.iterrows():
            key = f"{c.date[4:8]}{c.date[:4]}_{'abcdefgh'[c.cell - 1]}"
            hit = paper[ids == key]
            pairs_rows.append({"group": g, "how": "identifier", "date": c.date, "cell": c.cell, "paper_id": key,
                               "index": c["index"], "paper": float(hit.iloc[0, -1]) if len(hit) else np.nan})
    elif len(mine) == len(paper):
        for (_, c), p in zip(mine.iterrows(), pv):
            pairs_rows.append({"group": g, "how": "order", "date": c.date, "cell": c.cell, "paper_id": "",
                               "index": c["index"], "paper": p})
dsum = pd.DataFrame(rows)
print(dsum.round(3).to_string(index=False))
dpairs = pd.DataFrame(pairs_rows)
dpairs["rel_diff"] = (dpairs["index"] - dpairs.paper).abs() / dpairs.paper
print("\nMatched cells:", dpairs.paper.notna().sum(), "; median relative difference %.3f; within 10%%: %d; by group:"
      % (dpairs.rel_diff.median(), (dpairs.rel_diff <= 0.10).sum()))
print(dpairs.groupby(["group", "how"]).rel_diff.agg(["size", "median", "max"]).round(3).to_string())
print("Spearman correlation of matched values: %.3f" % stats.spearmanr(dpairs["index"], dpairs.paper, nan_policy="omit")[0])
print("Groups whose count equals the paper's n:", int((dsum.cells == dsum.paper_n).sum()), "of", len(dsum))

**Result: reproduced.** 89 of 92 cells have an index; three have no proximal and distal site of the same number. The counts equal the paper's n in 7 of 10 groups (iSPN on, iSPN on+sul and KO off are each one short). Over 63 matched cells the index differs from the paper's by a median of 4.0% (50 within 10%, Spearman correlation 0.911); the Figure 6D cells, matched by identifier, differ by medians of 3.5% (vehicle) and 6.4% (antagonist).

### link-sr-cells

> The dandiset's Sr2+-oEPSC recordings are cells of the paper's Figures 2 and 4, and their events
> agree with the paper's per-cell values.

In [ ]:
amp_t = pd.concat([tidy["Fig 2"]["Fig 2H_dSPN Sr-oEPSC amp"].set_axis(["id", "group", "amp"], axis=1),
                   tidy["Fig 4"]["Fig 4G_iSPN Sr-oEPSC amp"].set_axis(["id", "group", "amp"], axis=1)])
frq_t = pd.concat([tidy["Fig 2"]["Fig 2I_dSPN Sr-oEPSC freq"].set_axis(["id", "group", "freq"], axis=1),
                   tidy["Fig 4"]["Fig 4H_iSPN Sr-oEPSC freq"].set_axis(["id", "group", "freq"], axis=1)])
evt_t = pd.concat([tidy["Fig 2"]["Fig 2G_dSPN Sr-oEPSC histogram"].set_axis(["id", "group", "event", "amp"], axis=1),
                   tidy["Fig 4"]["Fig 4F_iSPN Sr-oEPSC histogram"].set_axis(["id", "group", "event", "amp"], axis=1)])
for t in (amp_t, frq_t, evt_t):
    t["id"] = t["id"].astype(str)

def sr_cell(s, threshold_sd=lid.SR_THRESHOLD_SD):
    evs = []
    for w in s["sweeps"]:
        onset = w["led_onset_s"] if w["led_trace_nonzero"] else s["epoch_stim_s"]
        evs.append(lid.detect_events(w["current"], w["rate"], onset, threshold_sd=threshold_sd))
    e = pd.concat(evs)
    return len(e), e.amp.mean(), len(e) / (0.36 * len(s["sweeps"])), float(np.median([x.attrs["noise"] for x in evs]))

rows = []
for s, r in zip(srs, sr_rows):
    n5, a5, f5, noise = sr_cell(s)
    n3, a3, f3, _ = sr_cell(s, 3.0)
    pe = evt_t[evt_t.id == r["paper_id"]]
    rows.append({"cell_type": r["cell_type"], "state": r["state"], "paper_id": r["paper_id"],
                 "in_paper": r["paper_id"] in set(amp_t.id), "noise_pA": noise,
                 "events": n5, "paper_events": len(pe), "mean_amp": a5, "paper_event_mean": pe.amp.mean(),
                 "paper_cell_amp": float(amp_t[amp_t.id == r["paper_id"]].amp.iloc[0]) if r["paper_id"] in set(amp_t.id) else np.nan,
                 "freq_Hz": f5, "paper_freq": float(frq_t[frq_t.id == r["paper_id"]].freq.iloc[0]) if r["paper_id"] in set(frq_t.id) else np.nan,
                 "events_3sd": n3, "mean_amp_3sd": a3})
sr_link = pd.DataFrame(rows)
print(sr_link.round(2).to_string(index=False))
print("\nPaper's frequency equals its event count / (0.36 s x 10 sweeps):",
      bool(np.allclose(sr_link.paper_freq, sr_link.paper_events / 3.6)))
ok_amp = (sr_link.mean_amp - sr_link.paper_event_mean).abs() <= 0.25 * sr_link.paper_event_mean
ok_n = (sr_link.events - sr_link.paper_events).abs() <= 0.25 * sr_link.paper_events
print(f"Cells matched by identifier: {sr_link.in_paper.sum()} of {len(sr_link)}; the paper has "
      f"{amp_t.id.nunique()} cells ({(amp_t.group.str.strip() == 'LID off').sum()} off, {(amp_t.group.str.strip() == 'LID on').sum()} on)")
print(f"Cells with mean amplitude within 25% of the paper's events: {ok_amp.sum()}; with event count within 25%: {ok_n.sum()}; both: {(ok_amp & ok_n).sum()} of {len(sr_link)}")
print("\nBy group (dandiset cells only; the paper's test cannot reach P < 0.05 with 3 against 4 cells):")
for ct in ("dSPN", "iSPN"):
    for col in ("mean_amp", "freq_Hz", "paper_cell_amp", "paper_freq"):
        a = sr_link[(sr_link.cell_type == ct) & (sr_link.state == "OffState")][col]
        b = sr_link[(sr_link.cell_type == ct) & (sr_link.state == "OnState")][col]
        print(f"  {ct} {col}: off {np.round(a.values, 2)} on {np.round(b.values, 2)}; MW P = {stats.mannwhitneyu(a, b).pvalue:.3f}")

**Result: partially reproduced.** All 14 cells match a paper identifier; the dandiset holds 14 of the paper's 33 cells, one per mouse. In every cell the detector here finds fewer events and a higher mean amplitude than the paper's events, and only 2 cells are within 25% on both, so the event detection differs from the paper's. With the paper's own per-cell amplitudes the 4 on-state dSPNs are all above the 3 off-state ones and the 4 on-state iSPNs all below the 3 off-state ones (MW P = 0.057 each, the smallest attainable).

### link-grab-rois

> The dandiset's GRAB-ACh recordings are regions of the paper's Figure 5F, and the normalised
> areas computed from them agree with the paper's per-region values.

In [ ]:
p5 = tidy["Fig 5"]["Fig 5F_GRAB-ACh_single-AUC"].set_axis(["roi", "group", "drug", "auc"], axis=1)
p5wide = p5.pivot_table(index=["group", "roi"], columns="drug", values="auc")
name = {"control": "No drug", "quinpirole": "quinpirole", "sulpiride": "sulpiride", "50nM_dopamine": "DA"}
pgroup = {"CTRL": "unlesioned", "PD": "6-OHDA", "OFF": "LID off"}
grows = []
for m in grabs:
    gm = lid.grab_measures(m)
    sp = gm[gm.stimulation == "single_pulse"].set_index("treatment").auc_norm.rename(index=name)
    cand = p5wide.loc[pgroup[m["state"]]][sp.index]
    dist = (cand - sp).abs()
    j = dist.sum(axis=1).idxmin()
    grows.append({"state": m["state"], "date": m["date"], **{f"{k}": round(v, 3) for k, v in sp.items()},
                  "best_roi": j, **{f"paper_{k}": round(cand.loc[j, k], 3) for k in sp.index},
                  "max_diff": round(dist.loc[j].max(), 3)})
glink = pd.DataFrame(grows)
print(glink.to_string(index=False))
print("\nRegions per group, dandiset against paper:",
      {pgroup[s]: (int((glink.state == s).sum()), int(p5wide.loc[pgroup[s]].shape[0])) for s in pgroup})
print("Mice with a match within 0.05 for every treatment:", int((glink.max_diff <= 0.05).sum()), "of", len(glink))
print("\nPer-mouse areas by treatment (the paper's tests cannot reach P < 0.05 with 3 mice per group or 3 or 5 pairs):")
print(glink.groupby("state")[["No drug", "quinpirole", "sulpiride", "DA"]].agg(lambda x: list(np.round(x.dropna(), 3))).to_string())

**Result: not reproduced.** The dandiset holds one region per mouse, 11 of the paper's 34 regions (3 of 13 unlesioned, 3 of 10 6-OHDA, 5 of 11 off-state). Only 5 of the 11 mice have a paper region within 0.05 on every treatment, so the regions cannot be identified in the paper's table.

### link-aim-mice

> The dandiset's AIM score tables are mice of the paper's Figures 8 and 10, with genotypes and
> scores that agree with the paper's tables.

In [ ]:
t8 = tidy["Fig 8"]["Fig 8B_total AIMs"].copy()
t8["animal"] = t8["Mouse ID"].astype(str).str.extract(r"(\d+)")[0].astype(int)
t10 = tidy["Fig 10"]["Fig 10B_total AIMs"].copy()
t10["animal"] = t10["Mouse ID"].astype(int)
paper_aim = pd.concat([t8.assign(fig="Fig 8", label=t8.Genotype.astype(str).str.strip())[["fig", "animal", "label", "Session#", "Total AIM score"]],
                       t10.assign(fig="Fig 10", label=t10.Group.astype(str).str.strip())[["fig", "animal", "label", "Session#", "Total AIM score"]]])
mine = aim.groupby(["animal_id", "session_number"]).agg(total=("total_score", "sum"), points=("total_score", "size")).reset_index()
rows = []
for a, g in mine.groupby("animal_id"):
    p = paper_aim[paper_aim.animal == a]
    lab = ", ".join(sorted(set(aim[aim.animal_id == a].genotype)))
    eq = [float(p[p["Session#"] == s]["Total AIM score"].iloc[0]) == t if (p["Session#"] == s).any() else None for s, t in zip(g.session_number, g.total)]
    rows.append({"animal": a, "in_paper": len(p) > 0, "paper_fig": p.fig.iloc[0] if len(p) else "", "paper_genotype": p.label.iloc[0] if len(p) else "",
                 "dandiset_table_genotypes": lab, "sessions": len(g), "totals": g.total.tolist(),
                 "paper_totals": p.sort_values("Session#")["Total AIM score"].tolist(), "sessions_equal": sum(e is True for e in eq)})
alink = pd.DataFrame(rows)
print(alink.to_string(index=False))
print("\nDandiset mice in the paper:", int(alink.in_paper.sum()), "of", len(alink), "; matched sessions equal:",
      int(alink.sessions_equal.sum()), "of", int(alink[alink.in_paper].sessions.sum()))
print("Paper mice: Fig 8", t8.animal.nunique(), "(", t8.groupby("Genotype").animal.nunique().to_dict(), "), Fig 10", t10.animal.nunique(),
      "(", t10.groupby("Group").animal.nunique().to_dict(), ")")
print("Paper's KO or deletion mice in the dandiset:", alink[alink.paper_genotype.isin(["KO", '"-M1R"'])].animal.tolist(),
      "; wild-type or control:", alink[alink.paper_genotype.isin(["WT", "control"])].animal.tolist())

**Result: not reproduced.** 7 of the 9 dandiset mice are in the paper's tables (4124 and 5871 are not). The two Figure 10 mice (5844 control, 5846 M1R deletion) have session totals equal to the paper's in all 10 sessions, but the five Figure 8 mice agree in only 5 of 25 sessions (mouse 1945, session 1, 33.5 here against 27.0). Of the paper's 23 Figure 8 mice the dandiset holds 4 KO and 1 wild type, and of its 21 Figure 10 mice one per group. The genotype labels in the dandiset contradict each other (data check); the paper's labels are the ones used.

## Claims on the dandiset: dSPNs (Figures 1 and 2)

### fig1f-dspn-rheobase-on-lower

> In dSPNs the rheobase is lower in the LID on-state than in the off-state.

In [ ]:
def by_mouse(df, value, groups):
    out = df[df.group.isin(groups)].groupby(["group", "date"])[value].agg(["size", "median"]).round(3)
    return out

off, on = som[som.group == "dSPN off"], som[som.group == "dSPN on"]
r = mw(off.rheobase, on.rheobase)
print("Rheobase (pA), off (a) against on (b):", show(r))
print("Off values:", sorted(off.rheobase.tolist())); print("On values:", sorted(on.rheobase.tolist()))
print("\nBy mouse:"); print(by_mouse(som, "rheobase", ["dSPN off", "dSPN on"]).to_string())
mo = som[som.group.isin(["dSPN off", "dSPN on"])].groupby(["group", "date"]).rheobase.mean().reset_index()
print("\nVariant, one value per mouse (mean rheobase):", show(mw(mo[mo.group == "dSPN off"].rheobase, mo[mo.group == "dSPN on"].rheobase)))

**Result: reproduced.** On-state median 140 pA against off-state 220 pA (15 cells each), MW P = 8.86e-06, effect r = -0.95. With one value per mouse (8 off, 10 on) P = 0.000425, and every on-state mouse is below every off-state mouse (r = -1.0).

### fig1e-dspn-fi-on-higher

> In on-state dSPNs suprathreshold current steps evoke more spikes than in off-state dSPNs across a
> range of intensities.

In [ ]:
def fi_sum(df):
    return df[amps].sum(axis=1, min_count=1)

r = mw(fi_sum(off), fi_sum(on))
print("Sum of spikes 20-300 pA, off (a) against on (b):", show(r))
complete = lambda df: df[df[amps].notna().all(axis=1)]
print("Variant, cells with every step only:", show(mw(fi_sum(complete(off)), fi_sum(complete(on)))))
print("\nMedian spike count by step (off / on) and per-step MW P:")
print(pd.DataFrame({a: [off[a].median(), on[a].median(), stats.mannwhitneyu(off[a].dropna(), on[a].dropna()).pvalue] for a in amps},
                   index=["off", "on", "P"]).round(3).to_string())

**Result: reproduced.** Summed spikes from 20 to 300 pA: median 88 on against 36 off, MW P = 8.8e-05 (P = 0.00015 for the 22 cells with every step). The on-state median is higher at every step from 120 to 300 pA, with per-step P at most 0.012.

### fig1f-dspn-sch-no-change

> Bath application of the D1R antagonist SCH 23390 does not reverse the lower rheobase of on-state
> dSPNs.

In [ ]:
sch = som[som.group == "dSPN on+SCH"]
r = mw(on.rheobase, sch.rheobase)
print("Rheobase, on (a) against on+SCH (b):", show(r))
print("On+SCH values:", sorted(sch.rheobase.tolist()))
paired = on.merge(sch, on=["date", "cell"], suffixes=("_on", "_sch"))
print(f"\nVariant, cells recorded in both conditions (n = {len(paired)}):")
print(paired[["date", "cell", "rheobase_on", "rheobase_sch"]].to_string(index=False))
w = stats.wilcoxon(paired.rheobase_on, paired.rheobase_sch) if (paired.rheobase_on != paired.rheobase_sch).any() else None
print("Wilcoxon:", None if w is None else f"statistic {w.statistic}, P = {w.pvalue:.3f}",
      "; median paired difference (SCH - on):", float((paired.rheobase_sch - paired.rheobase_on).median()))
print("\nBy mouse:"); print(by_mouse(som, "rheobase", ["dSPN on+SCH"]).to_string())

**Result: reproduced.** Rheobase on-state 140 pA (15 cells) against on-state with SCH 140 pA (10 cells), MW P = 1.0, effect r = 0.007. In the 5 cells recorded in both conditions the median paired difference is 0 pA (Wilcoxon P = 1.0).

### fig1i-dspn-dendritic-on-higher

> The dendritic excitability index of dSPNs is higher in the on-state than in the off-state.

In [ ]:
dend_nobg = cell_index(pd.concat([lid.dendritic_sites(d, 0.0, 0.0).assign(date=d["date"], cell_type=d["cell_type"]) for d in days if d["dendritic"]], ignore_index=True))
dend_strict = cell_index(sites, (0.8, 1.25))
dend_all = cell_index(sites, None)

def dend_test(ga, gb):
    out = {}
    for name, tab in [("main", dend), ("no background", dend_nobg), ("basal 0.8-1.25", dend_strict), ("all pairs", dend_all)]:
        out[name] = show(mw(tab[tab.group == ga]["index"], tab[tab.group == gb]["index"]))
    return pd.DataFrame(out).T

print("Index, off (a) against on (b):"); print(dend_test("dSPN off", "dSPN on").to_string())
print("\nOff:", np.round(sorted(dend[dend.group == "dSPN off"]["index"].dropna()), 3))
print("On: ", np.round(sorted(dend[dend.group == "dSPN on"]["index"].dropna()), 3))
print("\nBy mouse:"); print(by_mouse(dend, "index", ["dSPN off", "dSPN on"]).to_string())

**Result: reproduced.** Index on-state 0.476 against off-state 0.353 (10 cells each), MW P = 0.0091, effect r = 0.70. Without background subtraction P = 0.00728; keeping only pairs with a basal ratio between 0.8 and 1.25 (8 cells each) P = 0.0148.

### fig1i-dspn-dendritic-sch-lower

> SCH 23390 reverses the higher dendritic excitability of on-state dSPNs.

In [ ]:
print("Index, on (a) against on+SCH (b):"); print(dend_test("dSPN on", "dSPN on+SCH").to_string())
print("\nOn+SCH:", np.round(sorted(dend[dend.group == "dSPN on+SCH"]["index"].dropna()), 3))
print("Cells with an on-state index recorded in the SCH files:", int(((dend.group == "dSPN on") & dend.date.str.startswith("2019")).sum()))
print("\nBy mouse:"); print(by_mouse(dend, "index", ["dSPN on+SCH"]).to_string())

**Result: partially reproduced.** The criterion (P < 0.05) is not met, though the effect has the stated direction: on-state with SCH 0.419 (9 cells) against on-state 0.476 (10 cells), MW P = 0.0662, effect r = -0.51. With the stricter basal-ratio rule P = 0.524. The paper's median for the SCH group is 0.338 (link section), lower than the 0.419 here.

## Claims on the dandiset: iSPNs (Figure 3)

### fig3d-ispn-rheobase-on-higher

> In iSPNs the rheobase is higher in the on-state than in the off-state.

In [ ]:
ioff, ion, isul = som[som.group == "iSPN off"], som[som.group == "iSPN on"], som[som.group == "iSPN on+sul"]
print("Rheobase, off (a) against on (b):", show(mw(ioff.rheobase, ion.rheobase)))
print("Off:", sorted(ioff.rheobase.tolist())); print("On: ", sorted(ion.rheobase.tolist()))
print("\nBy mouse:"); print(by_mouse(som, "rheobase", ["iSPN off", "iSPN on"]).to_string())
mo = som[som.group.isin(["iSPN off", "iSPN on"])].groupby(["group", "date"]).rheobase.mean().reset_index()
print("\nVariant, one value per mouse:", show(mw(mo[mo.group == "iSPN off"].rheobase, mo[mo.group == "iSPN on"].rheobase)))

**Result: reproduced.** iSPN rheobase on-state 160 pA (11 cells) against off-state 120 pA (12 cells), MW P = 0.000829, effect r = 0.81. With one value per mouse (5 off, 8 on) P = 0.00899.

### fig3c-ispn-fi-on-lower

> The F-I relationship of iSPNs shifts to the right in the on-state.

In [ ]:
print("Sum of spikes 20-300 pA, off (a) against on (b):", show(mw(fi_sum(ioff), fi_sum(ion))))
print("Variant, cells with every step only:", show(mw(fi_sum(complete(ioff)), fi_sum(complete(ion)))))
print(pd.DataFrame({a: [ioff[a].median(), ion[a].median(), stats.mannwhitneyu(ioff[a].dropna(), ion[a].dropna()).pvalue] for a in amps},
                   index=["off", "on", "P"]).round(3).to_string())

**Result: reproduced.** Summed spikes 65 on against 118.5 off, MW P = 7.2e-05 (0.000165 for cells with every step). The on-state median is lower at every step from 120 to 300 pA, with per-step P at most 0.012.

### fig3d-ispn-sul-no-change

> Sulpiride does not alter the somatic excitability of on-state iSPNs.

In [ ]:
print("Rheobase, on (a) against on+sul (b):", show(mw(ion.rheobase, isul.rheobase)))
print("On+sul:", sorted(isul.rheobase.tolist()))
paired = ion.merge(isul, on=["date", "cell"], suffixes=("_on", "_sul"))
print(f"\nVariant, cells recorded in both conditions (n = {len(paired)}):")
print(paired[["date", "cell", "rheobase_on", "rheobase_sul"]].to_string(index=False))
print("By mouse:"); print(by_mouse(som, "rheobase", ["iSPN on+sul"]).to_string())

**Result: reproduced.** Rheobase on-state 160 pA (11 cells) against on-state with sulpiride 120 pA (7 cells), MW P = 0.143, effect r = -0.43. The difference leans toward a lower rheobase with sulpiride, as in the two cells recorded in both conditions (160 to 120 and 160 to 100 pA).

### fig3f-ispn-dendritic-on-lower

> The dendritic excitability index of iSPNs falls in the on-state compared with the off-state.

In [ ]:
print("Index, off (a) against on (b):"); print(dend_test("iSPN off", "iSPN on").to_string())
print("\nOff:", np.round(sorted(dend[dend.group == "iSPN off"]["index"].dropna()), 3))
print("On: ", np.round(sorted(dend[dend.group == "iSPN on"]["index"].dropna()), 3))
print("\nBy mouse:"); print(by_mouse(dend, "index", ["iSPN off", "iSPN on"]).to_string())

**Result: reproduced.** Index on-state 0.340 (9 cells) against off-state 0.524 (10 cells), MW P = 0.0128, effect r = -0.69. Without background subtraction P = 0.0128; with the stricter basal-ratio rule (9 cells each) P = 0.0774.

### fig3f-ispn-dendritic-sul-no-change

> Sulpiride does not significantly alter dendritic excitability in on-state iSPNs.

In [ ]:
print("Index, on (a) against on+sul (b):"); print(dend_test("iSPN on", "iSPN on+sul").to_string())
print("\nOn+sul:", np.round(sorted(dend[dend.group == "iSPN on+sul"]["index"].dropna()), 3))
print("By mouse:"); print(by_mouse(dend, "index", ["iSPN on+sul"]).to_string())

**Result: not reproduced.** The paper's test finds the difference the claim says is absent: on-state with sulpiride 0.506 (7 cells) against on-state 0.340 (9 cells), MW P = 0.0164, effect r = 0.71 (the paper reports P = 0.0545 with 8 and 10 cells). With the stricter basal-ratio rule P = 0.0663. The paper's own medians go the same way (0.499 and 0.345, link section).

## Claims on the dandiset: ACh release (Figure 5)

### fig5f-ach-off-elevated

> Evoked ACh release (GRAB-ACh3.0) in slices from LID off-state mice is greater than in unlesioned
> striatum.

In [ ]:
nod = glink.set_index("state")["No drug"]
print("Single-pulse normalised area, no drug, unlesioned (a) against off-state (b):", show(mw(nod["CTRL"], nod["OFF"])))
print("Unlesioned:", nod["CTRL"].round(3).tolist(), "; off-state:", nod["OFF"].round(3).tolist())
print("Smallest attainable two-sided P for 3 against 5 values: %.4f" % (2 / 56))
burst = []
for m in grabs:
    gm = lid.grab_measures(m)
    b = gm[(gm.stimulation == "burst_stimulation") & (gm.treatment == "control")]
    burst.append({"state": m["state"], "auc": float(b.auc_norm.iloc[0]) if len(b) else np.nan})
burst = pd.DataFrame(burst)
print("Variant, burst stimulation (fig. S1):", show(mw(burst[burst.state == "CTRL"].auc, burst[burst.state == "OFF"].auc)))
gd = []
for m in grabs:
    gm = lid.grab_measures(m)
    gd.append({"state": m["state"], "dff0": float(gm[(gm.stimulation == "single_pulse") & (gm.treatment == "control")].auc_dff0.iloc[0])})
gd = pd.DataFrame(gd)
print("Variant, unnormalised dF/F0 area (background 162):", show(mw(gd[gd.state == "CTRL"].dff0, gd[gd.state == "OFF"].dff0)))

**Result: reproduced.** Off-state 0.359 (5 mice) against unlesioned 0.112 (3 mice), MW P = 0.0357, the smallest attainable with these numbers: every off-state value is above every unlesioned one. Burst stimulation gives the same (P = 0.0357); the unnormalised dF/F0 area does not (P = 0.25).

### fig5f-ach-off-like-6ohda

> Evoked ACh release in the off-state is similar to that after a 6-OHDA lesion.

In [ ]:
print("Single-pulse normalised area, no drug, 6-OHDA (a) against off-state (b):", show(mw(nod["PD"], nod["OFF"])))
print("6-OHDA:", nod["PD"].round(3).tolist())
print("Variant, burst:", show(mw(burst[burst.state == "PD"].auc, burst[burst.state == "OFF"].auc)))

**Result: reproduced.** 6-OHDA 0.240 (3 mice) against off-state 0.359 (5 mice), MW P = 0.143, effect r = 0.73; with burst stimulation P = 1.0. With 3 and 5 mice only complete separation would be significant (P = 0.0357), so the test detects only a large difference.

## Claims on the dandiset: M1R and CDGI (Figures 6, 7 and 9)

### fig6c-m1r-antag-somatic-unchanged

> M1R antagonism throughout the off-state does not have a strong impact on the elevated somatic
> excitability of off-state iSPNs.

In [ ]:
veh, ant = som[som.group == "Fig6 vehicle"], som[som.group == "Fig6 M1R antagonist"]
print("Rheobase, vehicle (a) against M1R antagonist (b):", show(mw(veh.rheobase, ant.rheobase)))
print("Vehicle:", sorted(veh.rheobase.tolist()), "; antagonist:", sorted(ant.rheobase.tolist()))
print("Variant, sum of spikes 20-300 pA:", show(mw(fi_sum(veh), fi_sum(ant))))
print("By mouse:"); print(by_mouse(som, "rheobase", ["Fig6 vehicle", "Fig6 M1R antagonist"]).to_string())

**Result: reproduced.** Rheobase vehicle 120 pA (8 cells) against M1R antagonist 140 pA (10 cells), MW P = 0.681, effect r = 0.125; summed spikes P = 0.45.

### fig6d-m1r-antag-dendritic-lower

> M1R antagonism prevents the off-state elevation of iSPN dendritic excitability.

In [ ]:
print("Index, vehicle (a) against M1R antagonist (b):"); print(dend_test("Fig6 vehicle", "Fig6 M1R antagonist").to_string())
print("\nVehicle:", np.round(sorted(dend[dend.group == "Fig6 vehicle"]["index"].dropna()), 3))
print("Antagonist:", np.round(sorted(dend[dend.group == "Fig6 M1R antagonist"]["index"].dropna()), 3))
print("By mouse:"); print(by_mouse(dend, "index", ["Fig6 vehicle", "Fig6 M1R antagonist"]).to_string())

**Result: reproduced.** Index vehicle 0.451 (9 cells) against M1R antagonist 0.269 (7 cells), MW P = 0.0229, effect r = -0.68; with the stricter basal-ratio rule P = 0.0164.

### fig7c-cdgi-ko-somatic-oscillation

> Deletion of CDGI does not blunt the oscillation of iSPN somatic excitability between on- and
> off-states.

In [ ]:
koff, kon = som[som.group == "KO off"], som[som.group == "KO on"]
print("Rheobase, KO off (a) against KO on (b):", show(mw(koff.rheobase, kon.rheobase)))
print("KO off:", sorted(koff.rheobase.tolist()), "; KO on:", sorted(kon.rheobase.tolist()))
print("Variant, sum of spikes:", show(mw(fi_sum(koff), fi_sum(kon))))
print("By mouse:"); print(by_mouse(som, "rheobase", ["KO off", "KO on"]).to_string())

**Result: reproduced.** In CDGI KO iSPNs the rheobase is 210 pA on-state (8 cells) against 140 pA off-state (13 cells), MW P = 0.00104, effect r = 0.87; summed spikes P = 0.00335.

### fig7f-cdgi-ko-dendritic-unchanged

> The off-state elevation of iSPN dendritic excitability is absent in CDGI KO mice (off against on
> not different), while it is present in wild type.

In [ ]:
print("Index, KO off (a) against KO on (b):"); print(dend_test("KO off", "KO on").to_string())
print("\nKO off:", np.round(sorted(dend[dend.group == "KO off"]["index"].dropna()), 3))
print("KO on: ", np.round(sorted(dend[dend.group == "KO on"]["index"].dropna()), 3))
print("Wild type (fig3f):", show(mw(dend[dend.group == "iSPN off"]["index"], dend[dend.group == "iSPN on"]["index"])))
print("By mouse:"); print(by_mouse(dend, "index", ["KO off", "KO on"]).to_string())

**Result: reproduced.** KO index off-state 0.347 (10 cells) against on-state 0.286 (8 cells), MW P = 0.315, effect r = -0.30 (P = 0.601 with the stricter rule), while the wild-type difference is significant (P = 0.0128).

### fig9d-m1r-crispr-somatic-unchanged

> Deletion of M1Rs from iSPNs does not alter the elevated somatic excitability of off-state iSPNs.

In [ ]:
cc, cr = som[som.group == "Fig9 control"], som[som.group == "Fig9 M1R CRISPR"]
print("Rheobase, control (a) against M1R CRISPR (b):", show(mw(cc.rheobase, cr.rheobase)))
print("Control:", sorted(cc.rheobase.tolist()), "; CRISPR:", sorted(cr.rheobase.tolist()))
print("Variant, sum of spikes:", show(mw(fi_sum(cc), fi_sum(cr))))
print("By mouse:"); print(by_mouse(som, "rheobase", ["Fig9 control", "Fig9 M1R CRISPR"]).to_string())

**Result: reproduced.** Rheobase interleaved control 160 pA (6 cells) against M1R CRISPR 120 pA (8 cells), MW P = 0.147, effect r = -0.48; summed spikes P = 0.245.

## Claims on the paper's tables (data elsewhere)

These claims test data the dandiset does not hold, on the paper's per-cell and per-animal tables
(Zenodo doi:10.5281/zenodo.17307059). They say nothing about the dandiset.

### fig1-dspn-lesion-and-off-somatic

> dSPN somatic excitability (rheobase) rises after the 6-OHDA lesion and falls in the LID off-state.

In [ ]:
def tg(fig, sheet, group):
    return lid.tidy_group(tidy, (fig, sheet, group)).iloc[:, -1].astype(float)

print("Control (a) against 6-OHDA (b):", show(mw(tg("Fig 1", "Fig 1F_dSPN rheobase", "control"), tg("Fig 1", "Fig 1F_dSPN rheobase", "6-OHDA"))))
print("6-OHDA (a) against off (b):", show(mw(tg("Fig 1", "Fig 1F_dSPN rheobase", "6-OHDA"), tg("Fig 1", "Fig 1F_dSPN rheobase", "LID off"))))

**Result: reproduced.** On the paper's table, the rheobase is 120 pA after the lesion against 220 pA in controls (P = 0.000295) and 200 pA in the off-state against 120 pA after the lesion (P = 9.54e-05). The paper reports P < 0.0001 for both; the first is 0.000295 here.

### fig1-dspn-lesion-dendritic-unchanged

> The dSPN dendritic excitability index is not changed by the 6-OHDA lesion or in the off-state.

In [ ]:
sh = "Fig 1I_dSPN dendritic excitabil"
print("Control (a) against 6-OHDA (b):", show(mw(tg("Fig 1", sh, "control"), tg("Fig 1", sh, "6-OHDA"))))
print("6-OHDA (a) against off (b):", show(mw(tg("Fig 1", sh, "6-OHDA"), tg("Fig 1", sh, "LID off"))))

**Result: reproduced.** On the paper's table, control against 6-OHDA P = 0.82 and 6-OHDA against off-state P = 0.903 (the paper reports 0.824 and 0.905).

### fig3-ispn-lesion-and-off-somatic

> iSPN somatic excitability is lowered by the 6-OHDA lesion and restored in the LID off-state.

In [ ]:
sh = "Fig 3D_iSPN rheobase"
print("Control (a) against 6-OHDA (b):", show(mw(tg("Fig 3", sh, "Control"), tg("Fig 3", sh, "6-OHDA"))))
print("6-OHDA (a) against off (b):", show(mw(tg("Fig 3", sh, "6-OHDA"), tg("Fig 3", sh, "LID off"))))

**Result: reproduced.** On the paper's table, the rheobase is 200 pA after the lesion against 120 pA in controls (P = 0.000401) and 130 pA in the off-state (P = 0.000608 against 6-OHDA); the paper reports P = 0.002 for both.

### fig3-ispn-lesion-dendritic-up

> The iSPN dendritic excitability index rises after the lesion and stays raised in the off-state.

In [ ]:
sh = "Fig 3F_iSPN dendritic excitabil"
print("Control (a) against 6-OHDA (b):", show(mw(tg("Fig 3", sh, "Control"), tg("Fig 3", sh, "6-OHDA"))))
print("Control (a) against off (b):", show(mw(tg("Fig 3", sh, "Control"), tg("Fig 3", sh, "LID off"))))

**Result: reproduced.** On the paper's table, the index is 0.516 after the lesion and 0.519 in the off-state against 0.405 in controls (P = 0.0348 and 0.0242; the paper reports 0.0327 and 0.0220).

### fig7e-oxom-cdgi

> Oxotremorine-M increases bAP invasion of iSPN dendrites in wild-type but not in CDGI KO mice.

In [ ]:
t7 = tidy["Fig 7"]["Fig 7E_oxoM effect on DE"].set_axis(["expt", "genotype", "treatment", "index"], axis=1)
for gt in ("WT", "KO"):
    w = t7[t7.genotype == gt].pivot(index="expt", columns="treatment", values="index")
    res = stats.wilcoxon(w["before (control)"], w["oxoM"])
    print(f"{gt}: n = {len(w)}, median before {w['before (control)'].median():.3f}, after {w['oxoM'].median():.3f}, "
          f"increases {(w['oxoM'] > w['before (control)']).sum()}, Wilcoxon P = {res.pvalue:.4g}")

**Result: reproduced.** On the paper's table, oxo-M raises the index in wild type (8 of 10 increase, Wilcoxon P = 0.009766, as the paper reports) and not in KO (4 of 12, P = 0.4697; the paper reports 0.910).

### fig8a-cdgi-rotations

> Levodopa induces more contralateral rotations in CDGI KO mice than in wild type at the fifth dose.

In [ ]:
t = tidy["Fig 8"]["Fig 8A_rotations 5th"].set_axis(["mouse", "genotype", "rot"], axis=1)
print("Wild type (a) against KO (b):", show(mw(t[t.genotype.str.strip() == "WT"].rot, t[t.genotype.str.strip() == "KO"].rot)))

**Result: reproduced.** On the paper's table, 8.5 rotations in KO against 4.25 in wild type (12 and 10 mice), MW P = 0.012 (the text reports 0.0092).

### fig10a-m1r-rotations

> Levodopa induces more contralateral rotations in mice lacking M1Rs in iSPNs.

In [ ]:
t = tidy["Fig 10"]["Fig 10A_rotations in M1R CRISPR"].set_axis(["mouse", "group", "rot"], axis=1)
print("Control (a) against M1R deletion (b):", show(mw(t[t.group.str.strip() == "control"].rot, t[t.group.str.strip() == '"-M1R"'].rot)))

**Result: reproduced.** On the paper's table, 8.5 rotations with M1R deletion against 6.0 in controls, MW P = 0.00782 (the paper reports 0.0056).

### fig10ef-open-field

> In MCI-Park mice, a low dose of levodopa increases locomotion profoundly only when M1Rs are
> deleted from the striatum; M1R deletion alone does not.

In [ ]:
for sheet in ("Fig 10E_distance traveled", "Fig 10F_time mobile"):
    t = tidy["Fig 10"][sheet].set_axis(["mouse", "group", "v"], axis=1)
    t["group"] = t.group.astype(str).str.strip()
    gv = lambda g: t[t.group == g].v
    print(sheet)
    print("  control (a) against MCI-Park (b):", show(mw(gv("control"), gv("MCI-Park"))))
    print("  MCI-Park (a) against M1R deletion (b):", show(mw(gv("MCI-Park"), gv('"-M1R"'))))
    print("  MCI-Park (a) against levodopa (b):", show(mw(gv("MCI-Park"), gv('"+L-dopa"'))))
    print("  levodopa (a) against levodopa with deletion (b):", show(mw(gv('"+L-dopa"'), gv('"+L-dopa, -M1R"'))))
print(f"\nRun time {time.time() - started:.0f} s")

**Result: reproduced.** On the paper's tables, MCI-Park mice travel less than controls (4.68 against 10.35 m, P = 1.05e-05; time mobile P = 6.97e-06), M1R deletion alone does not change this (P = 0.628 and 0.467), and with levodopa the deletion raises distance from 6.52 to 28.09 m (P = 0.000332) and time mobile from 127 to 276 s (P = 0.000567).

## Summary

| Claim | Data | Outcome |
|---|---|---|
| link-somatic-cells | with inputs | partially reproduced |
| link-dendritic-cells | with inputs | reproduced |
| link-sr-cells | with inputs | partially reproduced |
| link-grab-rois | with inputs | not reproduced |
| link-aim-mice | with labels | not reproduced |
| fig1f-dspn-rheobase-on-lower | dandiset | reproduced |
| fig1e-dspn-fi-on-higher | dandiset | reproduced |
| fig1f-dspn-sch-no-change | dandiset | reproduced |
| fig1i-dspn-dendritic-on-higher | dandiset | reproduced |
| fig1i-dspn-dendritic-sch-lower | dandiset | partially reproduced |
| fig2h-dspn-sr-amplitude-on-larger | dandiset | blocked: other (too few cells) |
| fig2i-dspn-sr-frequency-unchanged | dandiset | blocked: other (too few cells) |
| fig2bc-dspn-spines | dandiset | blocked: not runnable |
| fig3d-ispn-rheobase-on-higher | dandiset | reproduced |
| fig3c-ispn-fi-on-lower | dandiset | reproduced |
| fig3d-ispn-sul-no-change | dandiset | reproduced |
| fig3f-ispn-dendritic-on-lower | dandiset | reproduced |
| fig3f-ispn-dendritic-sul-no-change | dandiset | not reproduced |
| fig4g-ispn-sr-amplitude-on-smaller | dandiset | blocked: other (too few cells) |
| fig4h-ispn-sr-frequency-unchanged | dandiset | blocked: other (too few cells) |
| fig4bc-ispn-spines | dandiset | blocked: not runnable |
| fig4k-confocal-spines | dandiset | blocked: not runnable |
| fig5f-ach-off-elevated | dandiset | reproduced |
| fig5f-ach-off-like-6ohda | dandiset | reproduced |
| fig5f-ach-6ohda-elevated | dandiset | blocked: other (too few mice) |
| fig5f-ach-d2-control | dandiset | blocked: other (too few mice) |
| fig5f-ach-da-suppresses | dandiset | blocked: other (too few mice) |
| fig6c-m1r-antag-somatic-unchanged | dandiset | reproduced |
| fig6d-m1r-antag-dendritic-lower | dandiset | reproduced |
| fig6f-m1r-antag-spines | dandiset | blocked: not runnable |
| fig7c-cdgi-ko-somatic-oscillation | dandiset | reproduced |
| fig7f-cdgi-ko-dendritic-unchanged | dandiset | reproduced |
| fig7h-cdgi-ko-spines | dandiset | blocked: not runnable |
| fig8bc-cdgi-aims | with labels | blocked: other (too few mice) |
| fig9d-m1r-crispr-somatic-unchanged | dandiset | reproduced |
| fig9f-m1r-crispr-spines | dandiset | blocked: not runnable |
| fig10b-m1r-aims | with labels | blocked: other (too few mice) |
| fig1-dspn-lesion-and-off-somatic | elsewhere | reproduced |
| fig1-dspn-lesion-dendritic-unchanged | elsewhere | reproduced |
| fig3-ispn-lesion-and-off-somatic | elsewhere | reproduced |
| fig3-ispn-lesion-dendritic-up | elsewhere | reproduced |
| fig7e-oxom-cdgi | elsewhere | reproduced |
| fig8a-cdgi-rotations | elsewhere | reproduced |
| fig10a-m1r-rotations | elsewhere | reproduced |
| fig10ef-open-field | elsewhere | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** Spike counts, rheobase, the dendritic index and the
GRAB-ACh area follow the paper's scripts, but the dendritic background is the script's constant for
every session (the files carry none), the basal-ratio rule for keeping dendritic pairs is a choice
made here (0.5 to 2; it drops no pair), and sites are paired by number. The GRAB-ACh stimulus time
comes from the trial table (3 s in six files where the script assumes 10 s). Sr2+-oEPSC events
were found with a threshold detector written here, which finds fewer, larger events than the
paper's TaroTools detection with visual verification. Mann-Whitney P values are computed with
scipy (exact without ties, asymptotic with ties), where Prism was used.

**What was found about the dandiset.**

- The somatic and dendritic patch-clamp data of Figures 1, 3, 6, 7 and 9 are there for every group
  except the unlesioned control and 6-OHDA groups, with the paper's cell counts; most cells' spike
  counts match the paper's table exactly. The groups must be read from `condition_subfolder`: files
  named for SCH 23390 or sulpiride hold baseline cells and cells recorded in both conditions.
- The Sr2+-oEPSC data hold one cell per mouse (14 of 33), the GRAB-ACh data one region per mouse
  (11 of 34), and the AIM data 9 mice, two of which are not in the paper; too few for the paper's
  tests. The AIM totals of the Figure 8 mice differ from the paper's.
- The AIM genotype labels contradict each other between the subject metadata, the session names,
  the table's genotype column and the video files.
- Spine density, rotation counts and open-field data are not in the dandiset as measurements; the
  spine and rotation raw data are (image stacks and videos), but the counts need manual software.
- The dendritic stimulus series are all zero, the Sr2+-oEPSC light-pulse series are zero in many
  sweeps and are not referenced by the recordings table, and the GRAB-ACh "zero-laser" background
  images are brighter than the signal.